In [185]:
import pandas as pd
from pathlib import Path
import ast
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## Ingestao dos dados

In [208]:
data_source = Path(r"C:\Users\dougl\OneDrive\Área de Trabalho\POS\pratica do que aprendi\modulo01-fundamentos-de-ia-e-llms-para-programadores\exemplo-01\Redes Neurais\data\Bank_churn.csv")


In [209]:
df = pd.read_csv(data_source)

## Exploracao dos dados

In [ ]:
# df.head()

In [ ]:
# df.columns

In [ ]:
# df["Geography:str,Gender:str"].head(10)

In [ ]:
# df["Exited"].value_counts()

In [ ]:
# df["Rating"].value_counts()

In [ ]:
# df.isnull().sum()

In [ ]:
# df["Geography:str,Gender:str"].value_counts()

In [ ]:
# df["Age"].describe()

In [ ]:
# df["EstimatedSalary"].describe()

In [ ]:
# df["Age"].head(20).tolist()

In [ ]:
# df["EstimatedSalary"].head(20).tolist()

In [ ]:
# df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
# df["EstimatedSalary"] = pd.to_numeric(df["EstimatedSalary"], errors="coerce")

In [ ]:
# df[["Age", "EstimatedSalary"]].dtypes

In [ ]:
# df.isnull().sum()

## Aqui comecamos a criar os dataframes de tranformacao

In [210]:
df_model = df[
    [
        "CreditScore",
        "Age",
        "Tenure",
        "Balance",
        "NumOfProducts",
        "HasCrCard",
        "IsActiveMember",
        "EstimatedSalary",
        "Geography:str,Gender:str",
        "Exited"
    ]
].copy()

In [211]:
ast.literal_eval(df_model["Geography:str,Gender:str"].iloc[0])

{'Geography': 'Paris-France', 'Gender': 'Male'}

In [212]:
df_model["Geography"]= df_model["Geography:str,Gender:str"].apply(
    lambda x: ast.literal_eval(x)["Geography"]
)

df_model["Gender"]= df_model["Geography:str,Gender:str"].apply(
    lambda x: ast.literal_eval(x)["Gender"]
)

In [213]:
df_model["Geography"].value_counts(dropna=False)

Geography
Paris-France     93778
Madrid-Spain     36050
Berlin-Gernay    34461
nan                745
Name: count, dtype: int64

In [214]:
df_model["Gender"].value_counts(dropna=False)

Gender
Male      88497
Female    68322
nan        8215
Name: count, dtype: int64

In [215]:
df_model["Gender"].isna().sum()

np.int64(0)

In [216]:
df_model["Gender"].dtype

<StringDtype(storage='python', na_value=nan)>

In [217]:
df_model["Gender"] = df_model["Gender"].replace("nan", pd.NA)
df_model["Geography"] = df_model["Geography"].replace("nan", pd.NA)

In [218]:
df_model[["Gender", "Geography"]].isna().sum()

Gender       8215
Geography     745
dtype: int64

In [219]:
df_model["Geography"] = df_model["Geography"].replace(
    "Berlin-Gernay", "Berlin-Germany"
)

In [220]:
df_model["Geography"].value_counts(dropna = False)

Geography
Paris-France      93778
Madrid-Spain      36050
Berlin-Germany    34461
NaN                 745
Name: count, dtype: int64

In [221]:
df_model["Gender"] = df_model["Gender"].fillna("Unknown")
df_model["Geography"] = df_model["Geography"].fillna("Unknown")

In [222]:
df_model = pd.get_dummies(
    df_model,
    columns = ["Geography", "Gender"]
)

In [223]:
df_model = df_model.drop(columns=["Geography:str,Gender:str"])

In [229]:
age_numeric = pd.to_numeric(
    df_model["Age"],
    errors="coerce"
)

In [234]:
age_clean = age_numeric.where(
    age_numeric.between(18, 100)
)

In [239]:
df_model["Age"] = age_clean

In [240]:
df_model["EstimatedSalary"] = pd.to_numeric(
    df_model["EstimatedSalary"],
    errors = "coerce"
)

In [241]:
x = df_model.drop(columns="Exited")
y = df_model["Exited"]

In [243]:
X_train, X_test, Y_train, Y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 42
)

In [244]:
imputer = SimpleImputer(strategy="median")

In [245]:
numeric_columns =[
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

In [246]:
imputer.fit(X_train[numeric_columns])

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['CreditScore','Age','Tenure',...,'HasCrCard','IsActiveMember', 'EstimatedSalary']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](8,)","[ 660. , 37. , 5. ,..., 1. , 0. ,117977.45]"


In [247]:
X_train[numeric_columns] = imputer.transform(
    X_train[numeric_columns]
)

In [248]:
X_test[numeric_columns] = imputer.transform(
    X_test[numeric_columns]
)

In [250]:
scaler = StandardScaler()

In [ ]:
scaler.fit(X_train[numeric_columns])

In [ ]:
X_train[numeric_columns] = scaler.transform(
    X_train[numeric_columns]
)

In [ ]:
X_train[numeric_columns].describe()

In [179]:
df_model["Age"].sort_values().head(10)

11312    -63000.0
89038    -63000.0
35491    -63000.0
117189   -61000.0
101572   -57000.0
26256    -52000.0
21455    -51000.0
148314   -50000.0
34694    -49000.0
48192    -49000.0
Name: Age, dtype: float64

In [ ]:
X_train["Age"].sort_values().head(10)


35491    -52.124692
11312    -52.124692
89038    -52.124692
117189   -50.470914
101572   -47.163358
26256    -43.028914
21455    -42.202025
148314   -41.375137
48192    -40.548248
34694    -40.548248
Name: Age, dtype: float64

In [180]:
df_model[df_model["Age"] < 0].value_counts().sort_index()

CreditScore  Age       Tenure  Balance    NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  Exited  Geography_Berlin-Germany  Geography_Madrid-Spain  Geography_Paris-France  Geography_Unknown  Gender_Female  Gender_Male  Gender_Unknown
418.0        -2900.0   4.0     0.00       2.0            0.0        1.0             88878.15         0       False                     False                   True                    False              False          True         False             1
459.0        -4800.0   1.0     0.00       1.0            1.0        0.0             191535.11        1       False                     False                   True                    False              True           False        False             1
485.0        -38000.0  2.0     101286.11  1.0            1.0        1.0             128891.71        0       False                     False                   True                    False              False          True         False             1
     

In [181]:
df[df["Age"].astype(str).str.contains("-", na=False)]["Age"].value_counts().head(20)

Age
-3900.0           9
-350.0            8
-3800.0           7
-34000.0          7
-3600.0           7
-290.0            7
-38000.0          7
37.0 year-old     7
29.0 years-old    7
-390.0            6
-3700.0           6
-43000.0          6
-400.0            6
33.0 year-old     5
35.0 year-old     5
38.0 years-old    5
41.0 years-old    5
40.0 year-old     4
-460.0            4
33.0 years-old    4
Name: count, dtype: int64